# 08 · Measure retrieval quality — the baseline

> **Run order.** This notebook is step 8 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


The number everything after this is measured against.

**Zero LLM calls.** Recall@k and MRR compare retrieved element IDs against the
ground-truth element IDs notebook 06 anchored — pure computation. That matters
practically as well as intellectually: the most iterative days of this project
cost nothing against any provider's rate limit.

A hit means a retrieved chunk contains one of the expected `element_id`s.
`pR@5` is the softer *page*-level recall: did we surface the right page at all?

Every run is **recorded**, not just printed. `analyst.evaluation` appends each one
to `results/runs.jsonl` with the config that produced it, a hash of the benchmark
it scored, and the git revision of the code — so "hybrid beat dense by X" is a
claim you can re-open six weeks later instead of re-running.

In [1]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from analyst import evaluation as ev
from analyst.config import get_settings
from analyst.embedding import MODELS
from analyst.retrievers import dense, open_store, store_for

settings = get_settings()
questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
print(f"{len(questions)} questions   benchmark {ev.bench_sha(questions)}   git {ev.git_rev()}")
pd.DataFrame([q.model_dump() for q in questions]).groupby(
    ["question_type", "match_kind"]).size().to_frame("n")

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


44 questions   benchmark 2c4aedf3dcb75f7e   git 3e65907-dirty

n
question_type match_kind    
growth        exact        9
              tolerant     1
value_lookup  exact       25
              tolerant     9

## One function, every configuration

The retriever is injected into `ev.evaluate`, so dense, hybrid and reranked
retrieval are scored by identical code on identical questions. That is the only
reason the deltas mean anything.

In [2]:
def score(model: str, filters: str = "ticker+year", deep: bool = False) -> ev.Run:
    """Evaluate one config, append it to the ledger, return the record."""
    embedder, store = open_store(settings, model)
    search = dense(embedder, store, filters)
    cfg = ev.RunConfig(retriever="dense", model=model, filters=filters,
                       limit=max(ev.K_VALUES), points=store.count())
    run = ev.build_run(
        cfg,
        ev.evaluate(questions, search, limit=max(ev.K_VALUES)),
        questions,
        # A second, wider pass: recall at depth is the ceiling for anything that
        # only reorders results, so it decides whether a reranker can help.
        deep=ev.evaluate(questions, search, limit=max(ev.DEPTHS)) if deep else None,
    )
    ev.append_run(run)
    return run

# Every model must be scored on the SAME number of chunks or the comparison is
# meaningless. A stopped index leaves a partial collection that looks fine.
FULL = store_for(settings, "bge-small").count()
print(f"complete index = {FULL:,} points")


complete index = 9,982 points

## Baseline: dense retrieval, metadata filters on

In [3]:
base = score("bge-small", filters="ticker+year", deep=True)
pd.DataFrame([base.row()])


C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


,run,retriever,model,filters,R@1,R@3,R@5,R@10,MRR,pR@5,points,p50_ms,bench,git
0,dense-bge-small-02c4b4ed,dense,bge-small,ticker+year,0.0227,0.0455,0.0455,0.0909,0.0392,0.0909,9982,84.8,2c4aedf3,3e65907-dirty


## Does metadata filtering earn its complexity?

Filtering restricts the candidate set *before* scoring. The claim is that this is
both faster and more accurate than filtering afterwards. Claims get measured here.

In [4]:
# Three policies, same collection, same questions. They are indistinguishable at
# k=10 and diverge only at depth - which is exactly what the headline table hides.
for policy in ("none", "ticker"):
    score("bge-small", filters=policy, deep=True)

pd.DataFrame([r.row() for r in ev.load_runs()])


C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


,run,retriever,model,filters,R@1,R@3,R@5,R@10,MRR,pR@5,points,p50_ms,bench,git
0,dense-bge-small-02c4b4ed,dense,bge-small,ticker+year,0.0227,0.0455,0.0455,0.0909,0.0392,0.0909,9982,84.8,2c4aedf3,3e65907-dirty
1,dense-bge-small-6cbee6af,dense,bge-small,none,0.0227,0.0455,0.0455,0.0909,0.0354,0.0909,9982,84.9,2c4aedf3,3e65907-dirty
2,dense-bge-small-73d5667f,dense,bge-small,ticker,0.0227,0.0455,0.0455,0.0909,0.0354,0.0909,9982,87.7,2c4aedf3,3e65907-dirty


## The measurement that decides Day 4

A cross-encoder reranker only reorders what retrieval already surfaced. Wherever
this curve goes flat is its hard ceiling, however good the reranker is.

In [5]:
pd.DataFrame([base.depth_curve], index=["recall"]).rename_axis("depth", axis=1)

depth,1,5,10,20,50,100,200
recall,0.0227,0.0455,0.0909,0.1364,0.2273,0.2727,0.4318


## ADR-006: which embedding model?

Every candidate in `analyst.embedding.MODELS` is indexed as its own collection by
notebook 07 and scored here on the same 44 questions. A model is chosen because it
won a measurement, not because of its reputation.

Collections that have not been indexed are skipped rather than failing the notebook.

In [6]:
for model in MODELS:
    if model == base.config.model:
        continue  # already scored above
    # Existence is a REST call; open_store would download the model first.
    store = store_for(settings, model)
    n = store.count() if store.exists() else 0
    if n != FULL:
        print(f"skip  {model:<12} {n:>6,} points (need {FULL:,}) - index incomplete")
        continue
    run = score(model, filters="ticker+year", deep=True)
    print(f"score {model:<12} R@5 {run.metrics.recall_at[5]:.3f}")

pd.DataFrame([r.row() for r in ev.load_runs()]).sort_values("R@5", ascending=False)


C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


score bge-base     R@5 0.091

score arctic-s     R@5 0.045

score minilm       R@5 0.068

,run,retriever,model,filters,R@1,R@3,R@5,R@10,MRR,pR@5,points,p50_ms,bench,git
3,dense-bge-base-b8c50be6,dense,bge-base,ticker+year,0.0000,0.0455,0.0909,0.1136,0.0303,0.1591,9982,396.1,2c4aedf3,3e65907-dirty
5,dense-minilm-d66ffa93,dense,minilm,ticker+year,0.0455,0.0455,0.0682,0.1136,0.0563,0.1136,9982,19.5,2c4aedf3,3e65907-dirty
1,dense-bge-small-6cbee6af,dense,bge-small,none,0.0227,0.0455,0.0455,0.0909,0.0354,0.0909,9982,84.9,2c4aedf3,3e65907-dirty
0,dense-bge-small-02c4b4ed,dense,bge-small,ticker+year,0.0227,0.0455,0.0455,0.0909,0.0392,0.0909,9982,84.8,2c4aedf3,3e65907-dirty
2,dense-bge-small-73d5667f,dense,bge-small,ticker,0.0227,0.0455,0.0455,0.0909,0.0354,0.0909,9982,87.7,2c4aedf3,3e65907-dirty
4,dense-arctic-s-d7b2c355,dense,arctic-s,ticker+year,0.0227,0.0455,0.0455,0.0455,0.0341,0.0455,9982,14.0,2c4aedf3,3e65907-dirty


## Where does it fail?

Aggregate numbers hide the interesting part.

In [7]:
worst = ev.evaluate(questions, dense(*open_store(settings, base.config.model), base.config.filters),
                    limit=max(ev.K_VALUES))
df = pd.DataFrame([r.model_dump() for r in worst])
print(df.groupby("question_type")["rank"].agg(
    n="size", found="count", best="min").to_string())

print(f"\nNever retrieved in the top {max(ev.K_VALUES)}: "
      f"{df['rank'].isna().sum()} of {len(df)}")
df[df["rank"].isna()].groupby(["ticker", "question_type"]).size().to_frame("misses")

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


                n  found  best
question_type                 
growth         10      0   NaN
value_lookup   34      4   1.0


Never retrieved in the top 10: 40 of 44

misses
ticker    question_type        
HDFCBANK  value_lookup        2
ICICIBANK growth              3
          value_lookup        6
RELIANCE  value_lookup        5
SUNPHARMA growth              7
          value_lookup       17

## The ledger

`results/leaderboard.md` is regenerated from `results/runs.jsonl` — committed, and
never edited by hand. Every future retrieval change appends to the same file, so
the delta is always one table away.

In [8]:
print(ev.write_leaderboard(ev.load_runs()))
print(ev.render_leaderboard(ev.load_runs()))

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\results\leaderboard.md

# Retrieval leaderboard

Generated from `results/runs.jsonl` by `analyst.evaluation`. Never edit by hand.

> Ground truth is **single-anchor**: each question names one element holding the
> answer, so a different page that also states it scores as a miss. Every row is
> strict the same way, so the deltas are fair; no number here is absolute quality.

| run | retriever | model | filters | R@1 | R@3 | R@5 | R@10 | MRR | pR@5 | p50 ms | bench | git |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| dense-bge-base-b8c50be6 | dense | `bge-base` | `ticker+year` | 0.000 | 0.045 | 0.091 | 0.114 | 0.030 | 0.159 | 396 | `2c4aedf3` | `3e65907-dirty` |
| dense-minilm-d66ffa93 | dense | `minilm` | `ticker+year` | 0.045 | 0.045 | 0.068 | 0.114 | 0.056 | 0.114 | 20 | `2c4aedf3` | `3e65907-dirty` |
| dense-bge-small-02c4b4ed | dense | `bge-small` | `ticker+year` | 0.023 | 0.045 | 0.045 | 0.091 | 0.039 | 0.091 | 85 | `2c4aedf3` | `3e65907-dirty` |
| dense-bge-small-6cbee6af | dense | `bge-small`